## Sammanfattning

Ett logistikoperationsteam planerar ett randomiserat fältförsök som jämför tre förarruttningsstrategier (statisk baslinje, dynamisk omdirigering, AI-optimerad) över tre depåregioner, med genomsnittlig leveransförsening (minuter) som svarsvariabel. PROC GLMPOWER tar en *exempel*-datamängd med konjekturerade cellmedelvärden och löser för det totala antalet förare som behövs för att detektera strategieffekter vid 80 % och 90 % styrka, och kartlägger sedan hur den uppnåeliga styrkan urholkas när rutt-till-rutt-variabiliteten ökar.

# Dimensionering av ett förarruttningsförsök med PROC GLMPOWER

## Sammanfattning

Ett logistikoperationsteam ska snart lansera ett randomiserat fältförsök som jämför tre förarruttningsstrategier -- en **statisk** baslinje, ett **dynamiskt** omdirigeringssystem och en **AI-optimerad** planerare -- utrullade över tre depåregioner (Nordöst, Mellanvästern, Väst). Svarsvariabeln är genomsnittlig **leveransförsening i minuter**. Innan teamet binder flottkapacitet till försöket måste de svara på: *hur många förare behöver vi för att tillförlitligt detektera den operativa förbättring vi förväntar oss?*

Denna notebook använder **PROC GLMPOWER** för att utföra prospektiv styrke- och stickprovsstorleksanalys för den generella linjära modellen bakom försöket. Utifrån en *exempel*-datamängd med konjekturerade cellmedelvärden (1) löser den för den totala inskrivningen som behövs för att nå 80 % och 90 % styrka för de övergripande strategi- och regioneffekterna, (2) kartlägger hur uppnåelig styrka försämras när rutt-till-rutt-variabiliteten stiger, och (3) producerar en styrkekurva som stöd för inskrivningsbeslutet.

> **Viktig slutsats:** Med en rimlig felstandardavvikelse på 8 minuter ger ungefär **63 förare** 80 % styrka och **83 förare** ger 90 % styrka för att detektera ruttstrategieffekter -- men om förseningsvariabiliteten ligger närmare 10 minuter når inte ens 90 inskrivna förare 70 % styrka, vilket understryker värdet av tighta, väl instrumenterade rutter.

---
## 1. Bygg den exemplariska designen

Det första steget kodar försökets layout och teamets *konjekturerade* medelförsening för varje kombination av ruttstrategi x depåregion. Vi fixerar ett slumptalsfrö och lägger till en försumbar brusning så att medelvärdena ser realistiska ut samtidigt som den balanserade 3x3-strukturen bevaras. `cell_n`-vikten på 1 i varje cell talar om för GLMPOWER att designen är balanserad.

In [1]:
/* ================================================================
   Generera exempeldatamängden med konjekturerade medelförseningar.
   En rad per design-cell för ruttstrategi x depåregion.
   ================================================================ */
data routing_trial;
   CALL streaminit(20260531);
   LÄNGD routing_strategy $8 depot_region $9;
   FÄLT strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   FÄLT region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   FÄLT smean[3]     _temporary_ (18.0 14.5 11.0);   /* konjekturerade strategimedelvärden */
   FÄLT radj[3]      _temporary_ (1.5  0.0 -1.0);    /* regionala justeringar (min)  */
   GÖR i = 1 TILL 3;
      GÖR j = 1 TILL 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         UTDATA;
      SLUT;
   SLUT;
   TA_BORT i j;
KÖR;

PROCEDUR SKRIV data=routing_trial ETIKETT noobs;
   VARIABEL routing_strategy depot_region mean_delay_min cell_n;
   ETIKETT routing_strategy="Ruttstrategi" depot_region="Depåregion"
         mean_delay_min="Genomsnittlig leveransförsening (min)" cell_n="Cellvikt";
   TITEL "Exempelcellmedelvärden: konjekturerad leveransförsening (minuter)";
KÖR;


                           Exempelcellmedelvärden: konjekturerad leveransförsening (minuter)                            

Ruttstrategi   Depåregion   Genomsnittlig leveransförsening (min)  Cellvikt
Static        Northeast                              19.687507651         1
Static        Midwest                               17.9871067398         1
Static        West                                  16.8240210086         1
Dynamic       Northeast                             15.9537535365         1
Dynamic       Midwest                               14.4415169858         1
Dynamic       West                                  12.8636389493         1
AIOpt         Northeast                             12.6143811724         1
AIOpt         Midwest                                10.893885771         1
AIOpt         West                                    9.635351403         1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Stickprovsstorlek för den övergripande designen

Med designen på plats ber vi GLMPOWER att **lösa för total stickprovsstorlek** (`NTOTAL = .`) vid 80 % och 90 % styrka. `MODEL`-satsen anger en tvåvägslayout med interaktion (`routing_strategy*depot_region`), `WEIGHT cell_n` deklarerar de balanserade profilvikterna, och `STDDEV = 8` är det antagna medelkvadratfelet för leveransförsening. `NFRACTIONAL` låter proceduren rapportera exakta fraktionella antal före upprundning.

Vi förregistrerar också tre planerade `CONTRAST`-jämförelser -- AI-optimerad mot statisk, dynamisk mot statisk, och valfri omdirigering mot statisk -- vilka dokumenterar de operativt meningsfulla hypoteser försöket är byggt för att testa.

In [2]:
/* ================================================================
   Lös för det totala antalet förare som behövs för att nå 80 % och
   90 % styrka för ruttstrategi- och depåregioneffekterna.
   ================================================================ */
PROCEDUR glmpower data=routing_trial;
   KLASS routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   VIKT cell_n;
   ETIKETT routing_strategy="Ruttstrategi" depot_region="Depåregion"
         mean_delay_min="Genomsnittlig leveransförsening (min)";
   CONTRAST "AI-optimerad mot statisk"     routing_strategy -1  0  1;
   CONTRAST "Dynamisk mot statisk"         routing_strategy -1  1  0;
   CONTRAST "Valfri omdirigering mot statisk" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TITEL "Stickprovsstorlek för att detektera ruttstrategieffekter på förseningen";
KÖR;


                           Exempelcellmedelvärden: konjekturerad leveransförsening (minuter)                            

The GLMPOWER Procedure


                 Fixed Scenario Elements                  

Item                Value                                 
------------------  --------------------------------------
Dependent Variable  Genomsnittlig leveransförsening (min) 
Source              Ruttstrategi                          
Source              Depåregion                            
Source              Ruttstrategi*Depåregion               

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Styrkans känslighet för variabilitet och försöksstorlek

Svaret på stickprovsstorleken hänger på den antagna felstandardavvikelsen, som sällan är känd exakt i förväg. Här vänder vi på frågan: vi **fixerar** flera kandidatinskrivningstotaler (`NTOTAL = 45 90 180`) och **löser för uppnådd styrka** (`POWER = .`) över ett rutnät av rimliga förseningsstandardavvikelser (6, 8 och 10 minuter). Den resulterande tabellen är en känslighetskarta -- den visar hur robust varje inskrivningsplan är om verklighetens ruttvariabilitet visar sig högre än man hoppats.

In [3]:
/* ================================================================
   Känslighetsrutnät: uppnådd styrka över kandidatförsöksstorlekar
   och rimliga förseningsstandardavvikelser.
   ================================================================ */
PROCEDUR glmpower data=routing_trial;
   KLASS routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   VIKT cell_n;
   ETIKETT routing_strategy="Ruttstrategi" depot_region="Depåregion"
         mean_delay_min="Genomsnittlig leveransförsening (min)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TITEL "Uppnådd styrka över variabilitets- och försöksstorleksscenarier";
KÖR;


                           Exempelcellmedelvärden: konjekturerad leveransförsening (minuter)                            

The GLMPOWER Procedure


                 Fixed Scenario Elements                  

Item                Value                                 
------------------  --------------------------------------
Dependent Variable  Genomsnittlig leveransförsening (min) 
Source              Ruttstrategi                          
Source              Depåregion                            

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Styrkekurva för inskrivningsbeslutet

Slutligen spårar vi en **styrkekurva** -- uppnådd styrka när den totala inskrivningen växer från 30 till 270 förare i steg om 30 -- för strategi-plus-region-modellen vid den förväntade standardavvikelsen på 8 minuter. Att lösa `POWER = .` över det inskrivningsrutnätet producerar kurvan som en tabellerad `N Total`-mot-`Power`-serie, så vi kan läsa av inskrivningen där vart och ett av de konventionella 80 %- och 90 %-målen passeras.

In [4]:
/* ================================================================
   Styrkekurva: uppnådd styrka vs. totalt antal inskrivna förare,
   sveper från 30 till 270 i steg om 30 vid den förväntade
   standardavvikelsen på 8 min.
   ================================================================ */
PROCEDUR glmpower data=routing_trial;
   KLASS routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   VIKT cell_n;
   ETIKETT routing_strategy="Ruttstrategi" depot_region="Depåregion"
         mean_delay_min="Genomsnittlig leveransförsening (min)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TITEL "Styrkekurva: uppnådd styrka vs. totalt antal inskrivna förare";
KÖR;


                           Exempelcellmedelvärden: konjekturerad leveransförsening (minuter)                            

The GLMPOWER Procedure


                 Fixed Scenario Elements                  

Item                Value                                 
------------------  --------------------------------------
Dependent Variable  Genomsnittlig leveransförsening (min) 
Source              Ruttstrategi                          
Source              Depåregion                            

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Tolkning och rekommendation

Analysen ger operationsteamet en försvarbar inskrivningsplan:

- **Baslinjedimensionering (avsnitt 2).** Med antagandet om en felstandardavvikelse på 8 minuter når den fullständiga tvåvägsmodellen (strategi, region och deras interaktion) **80 % styrka vid 63 förare** och **90 % styrka vid 83 förare**. Om man rundar upp för bortfall ligger ett inskrivningsmål nära **90 förare** bekvämt över 90 %-tröskeln.

- **Känsligheten spelar roll (avsnitt 3).** Styrkan är mycket känslig för förseningsvariabilitet. Vid 90 förare faller uppnådd styrka från ~99 % (SD=6) till ~87 % (SD=8) till ~68 % (SD=10). Ett pilotförsök med 45 förare är endast tillräckligt om variabiliteten är låg (~81 % styrka vid SD=6) och är kraftigt underdimensionerat (~37 %) om SD når 10. Den praktiska innebörden: att investera i konsekventa, väl instrumenterade rutter för att hålla nere variabiliteten är lika värdefullt som att lägga till förare.

- **Styrkekurvan (avsnitt 4).** Spårad för strategi-plus-region-modellen (utan interaktionsterm, den lins som används för känslighetssvepet), klättrar uppnådd styrka från 0,37 vid 30 förare till 0,69 vid 60, 0,87 vid 90 och 0,95 vid 120, och planar ut över 0,99 vid 180. Om man läser kurvan mot målen nås 80 % styrka nära 77 förare och 90 % nära 99 -- något högre än dimensioneringen för den fullständiga modellen i avsnitt 2 eftersom borttagandet av interaktionstermen sprider samma effekt över färre frihetsgrader i modellen.

**Rekommendation:** Skriv in ungefär **90 förare** (30 per ruttstrategi, balanserat över de tre depåregionerna). Detta klarar 90 % styrka för den fullständiga modellen vid den förväntade standardavvikelsen på 8 minuter, håller ~87 % styrka även på den mer konservativa reducerade modellkurvan, och håller försöket tillräckligt litet för att genomföra inom ett enda driftskvartal.

*Obs:* GLMPOWER analyserar den konjekturerade *designen*, inte observerade utfall -- så trovärdigheten i dessa siffror vilar på de konjekturerade medelvärdena och standardavvikelsen. Team bör se över dimensioneringen så snart en kort pilot ger en empirisk uppskattning av leveransförseningens variabilitet.